In [139]:
!pip install -q pytorch-lightning datasets transformers evaluate accelerate scikit-learn

In [140]:
# Instalacion de librerias
import random
import torch
import numpy as np
import os
from pytorch_lightning import seed_everything
import matplotlib.pyplot as plt
import seaborn as sns
import re

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)# Store the average loss after eachepoch so we can plot them.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["TF_DETERMINISTIC_OPS"] = "1" # See:https://github.com/NVIDIA/tensorflow-determinism#confirmed-current-gpu-specific-sources-of-non-determinism-with-solutions
seed_everything(42, workers=True)

from datasets import Dataset, DatasetDict #, load_metric EN PRINCIPIO ESTÁ DESCONTINUADO, TENDREMOS QUE BUSCAR OTRA ALTERNATIVA
import pandas as pd
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from huggingface_hub import login

INFO:lightning_fabric.utilities.seed:Seed set to 42


In [141]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

✅ GPU detectada. Trabajando con: "NVIDIA L4"
Memoria VRAM total disponible: 23.66 GB


In [142]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Entrenamiento Train/Valid/Test

## Lectura y etiquetado de Datos

In [143]:
# --- 1. PARAMETRIZACIÓN DE LA ETIQUETA ---
# Cambia esta variable para entrenar un modelo distinto cada vez
#ETIQUETA_OBJETIVO = "IDEOLOGICAL-INEQUALITY"
#ETIQUETA_OBJETIVO = "STEREOTYPING-DOMINANCE"
#ETIQUETA_OBJETIVO = "OBJECTIFICATION"
#ETIQUETA_OBJETIVO = "SEXUAL-VIOLENCE"
ETIQUETA_OBJETIVO = "MISOGYNY-NON-SEXUAL-VIOLENCE"

print(f"--- Preparando datos para la etiqueta: {ETIQUETA_OBJETIVO} ---")

--- Preparando datos para la etiqueta: MISOGYNY-NON-SEXUAL-VIOLENCE ---


In [144]:
# --- 2. NUEVAS RUTAS TAREA 3.3 ---
RUTA_TRAIN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_train_3_3.csv"
RUTA_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv"

print("Cargando el dataset maestro de entrenamiento y test...")
df_train_master = pd.read_csv(RUTA_TRAIN)
test_df = pd.read_csv(RUTA_TEST)

Cargando el dataset maestro de entrenamiento y test...


In [145]:
# --- 3. ADAPTACIÓN PARA HUGGING FACE ---
# Renombramos la columna objetivo a 'label' para que el Trainer la detecte automáticamente
df_train_master = df_train_master.rename(columns={ETIQUETA_OBJETIVO: "label"})
test_df = test_df.rename(columns={ETIQUETA_OBJETIVO: "label"})

# Asegurarnos de que son enteros
df_train_master["label"] = df_train_master["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

In [146]:
# --- 4. DIVISIÓN TRAIN/VALIDATION ---
# 90% Train, 10% Validation estratificado sobre la etiqueta elegida
train_df, val_df = train_test_split(
    df_train_master,
    test_size=0.10,
    stratify=df_train_master["label"],
    random_state=42
)

print("\nDistribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())

print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

print("\nDistribución fichero de test (estático):")
print(test_df['label'].value_counts())

# Identificador dinámico para guardar el modelo
resultado = f"Roberta_Task3.3_{ETIQUETA_OBJETIVO}"
print(f"\nIdentificador del modelo a guardar: {resultado}")


Distribución fichero de entrenamiento (Train):
label
0    739
1    135
Name: count, dtype: int64

Distribución fichero de validación (Valid):
label
0    83
1    15
Name: count, dtype: int64

Distribución fichero de test (estático):
label
0    189
1     41
Name: count, dtype: int64

Identificador del modelo a guardar: Roberta_Task3.3_MISOGYNY-NON-SEXUAL-VIOLENCE


## Preprocesado del texto

In [147]:
# Para ver el vocabulario del dataset

import pandas as pd

# Suponiendo que la columna de texto se llama 'text'
# Tokeniza y obtiene todas las palabras
vocabulario = set(
    palabra.lower()
    for fila in df_train_master["text"].dropna()
    for palabra in fila.split()
)

# Mostrar todo el vocabulario
print(vocabulario)

# (Opcional) Ver el número de palabras únicas
print(f"\nTotal de palabras únicas: {len(vocabulario)}")


{'excel', 'will.', 'bra,', 'gringa', 'nerves.', 'tra-tra-travetrabajó', 'asegurar', 'stand-up', 'plaid', 'además', 'happily', 'realidad,', 'autistic', 'desconfiados', 'contar', 'why', 'hazme', 'pegó', 'stayed', 'offend', 'hold', 'mishmmii', 'pay', 'arruinar', 'ddp', 'lobo,', 'sí.¿', 'taught', 'enoovs', 'hombre.el', 'voluntaria.', 'sábado,', 'davidkasprak', 'lleno', 'edad', 'incluye', 'irresistible.', 'matrioshka', 'atómico"', 'king', 'begging,', 'awesome.', 'pecadores!', 'anduviera', 'out,', '70%', 'fotos.', 'sociedad,', 'subamos!', 'tontería.otra', 'stop', 'justificación.', 'euros?', 'om', 'tell', '"transgénero"', '"y', 'trepat', 'ride.', 'pibinos', 'final', '.riverau', 'publicitaria', 'vida', 'enojada.', '1504', 'honk,', 'process.', 'miren,', 'hiding', 'v1de0', 'quedaste', '0.08', 'quieran', 'peso.', 'esas', 'miles', 'rosas.', 'mando', 'esta,', 'entiendo', 'banda.', 'actually,', 'supuesto', '(8m', '27', 'omitiste', 'preventiva', 'daten', 'speak,', "0'", 'cuerpos', 'vejiga', 'swimsuit

In [148]:
import re

def remove_links(tweet):
    """Takes a string and removes web links from it"""
    tweet = re.sub(r'http\S+', '', tweet)        # remove http links
    tweet = re.sub(r'bit.ly/\S+', '', tweet)     # remove bitly links
    tweet = re.sub(r'\[link\]', '', tweet )      # remove [link]
    tweet = re.sub(r'\[url\]', '', tweet )       # remove [url]
    tweet = re.sub(r'pic.twitter\S+','', tweet)
    return tweet

def remove_users(tweet):
    """Takes a string and removes retweet and @user information"""
    tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet
    tweet = re.sub('(@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove tweeted at
    tweet = re.sub(r'\[user\]', '', tweet )                      # remove [user]
    return tweet

def remove_hashtags(tweet):
    """Takes a string and removes any hash tags"""
    tweet = re.sub('(#[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove hash tags
    return tweet

def remove_av(tweet):
    """Takes a string and removes AUDIO/VIDEO tags or labels"""
    tweet = re.sub('VIDEO:', '', tweet)  # remove 'VIDEO:' from start of tweet
    tweet = re.sub('AUDIO:', '', tweet)  # remove 'AUDIO:' from start of tweet
    return tweet

def remove_emojis(tweet):
    emoj = re.compile("["
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U0001F300-\U0001F5FF"  # Miscellaneous Symbols And Pictographs
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00010000-\U0010FFFF"
        u"\U0001F680-\U0001F6FF"  # Transport and Map Symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\ufe0f"  # dingbats

                      "]+", re.UNICODE)
    return re.sub(emoj, '', tweet)

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_7906/1710960392.py:14: SyntaxWarning: invalid escape sequence '\s'
  tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet


Convertimos todo a minúsculas

In [149]:
campo_texto = 'text'

train_df[campo_texto] = train_df[campo_texto].str.lower()
val_df[campo_texto] = val_df[campo_texto].str.lower()
test_df[campo_texto] = test_df[campo_texto].str.lower()

## Definición de métricas

In [150]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

## Entrenamiento del modelo

In [151]:
# Cargamos el token de HuggingFace que lo tenemos en un fichero oculto e iniciamos sesión

#with open("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/huggingfaceToken.txt", "r") as file:
#    hf_token = file.read().strip()

from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)
print("Login completado con éxito")

# ruta_archivo = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/token_hf.txt'

# try:
#     # Abrimos el archivo en modo lectura ('r)
#     with open(ruta_archivo, 'r') as file:
#         hf_token = file.read().strip()

#         # Hacemos el login
#         login(token=hf_token)
#         print("Login completado con éxito")
# except:
#     print("Error: No se ha encontrado el archivo " + ruta_archivo + " en la ruta especificada")

Login completado con éxito


In [152]:
# Elección del modelo

model_checkpoint = 'FacebookAI/roberta-base'

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [153]:
# Se carga el modelo preentrenado
n_labels = 2

# El uso de una función de inicialización facilita la repetición del entrenamiento
# Se puede usar la misma función de inicialización en diferentes ejecuciones del código o en configuraciones de entrenamiento diferentes
# Esto facilita la repetición del entrenamiento y la reproducibilidad, ya que se puede inicializar el modelo
# de la misma manera en cada ejecución.

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint,
                                                              num_labels = n_labels) #, return_dict = True )
                                                              # use_auth_token = 'token propio de HugginFace')

In [154]:
# Para saber el nombre del modelo
model_name = model_checkpoint.split("/")[-1]
model_name

'roberta-base'

## Fine-tuning

In [155]:
# Selección de hiperparámetros
#BATCH_SIZE = 32
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 15
#LEARNING_RATE = 3e-5
LEARNING_RATE = 1e-5
MAX_LENGTH = 128
WEIGHT_DECAY = 0.1

In [156]:


def tokenize_data(examples):
  return tokenizer(examples[campo_texto], truncation=True, max_length=MAX_LENGTH, padding=True)

In [157]:
from transformers import Trainer, TrainingArguments

optim = ["adamw_hf", "adamw_torch", "adamw_apex_fused", "adafactor", "adamw_torch_xla"]

training_args = TrainingArguments(
    output_dir = 'results',
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1', # Cambiar la metrica por la que queremos ajustar
    weight_decay = WEIGHT_DECAY,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    save_total_limit = 3,
    optim = optim[1],
    push_to_hub = False,
    greater_is_better = True,
    logging_strategy = 'epoch'
)

#output_dir = '/home/adrian/Escritorio/DeepSexist/TrainingBooks/Task3.1/Research/ModelosTransformers'
#output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Perspectivismo/' + resultado
output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/' + ETIQUETA_OBJETIVO + "/" + resultado
model_output_dir = f"{output_dir}/modelo_{model_name}"

# Convierte directamente los DataFrames de Pandas a Datasets de Hugging Face
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

# Reseteamos el formato para que no haya fallos
train_dataset.reset_format()
valid_dataset.reset_format()

# Obtenemos los nombres de todas las columnas originales
columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names

#Protegemos la columna "label" para no eliminarla accidentalmente

columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train:
    columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid:
    columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

# Redefinimos model_init aquí para aplicar la congelación (Layer Freezing)
# def model_init():
#     # 1. Cargamos el modelo base
#     model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)
#     # 2. Congelamos todas las capas base (el "cerebro")
#     for param in model.base_model.parameters():
#         param.requires_grad = False
#     # El classifier sigue estando activo por defecto
#     return model

# Descongelación parcial de las últimas 3 capas

# def model_init():
#     # 1. Cargamos el modelo base
#     model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)

#     # 2. Congelamos TODO el cerebro base por defecto
#     for param in model.base_model.parameters():
#         param.requires_grad = False

#     # 3. Buscamos dónde guarda el modelo sus capas (esto varía según el modelo que estemos trabajando)
#     if hasattr(model.base_model, 'encoder') and hasattr(model.base_model.encoder, 'layer'):
#         capas_transformers = model.base_model.encoder.layer
#     elif hasattr(model.base_model, 'layers'):
#         capas_transformers = model.base_model.layers
#     else:
#         print("No se encontró la estructura de capas estándar. Se entrenará solo con la cabeza.")
#         capas_transformers=[]

#     # 4. Descongelamos solo las últimas 3 capas para que se adapten al vocabulario (TikTok)
#     capas_a_descongelar = 3
#     if len(capas_transformers) >= capas_a_descongelar:
#         print("Descongelando las últimas " + str(capas_a_descongelar) + " capas de un total de " + str(len(capas_transformers)))
#         for capa in capas_transformers[-capas_a_descongelar:]:
#             for param in capa.parameters():
#                 param.requires_grad = True
#     return model

# Inicializa el entrenador
trainer = Trainer(
    model_init = model_init,
    args=training_args,
    compute_metrics = compute_metrics,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    processing_class=tokenizer,
)

# Entrena el modelo
trainer.train()

# Guarda el modelo entrenado
trainer.save_model(model_output_dir)

Map:   0%|          | 0/874 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Resultados contra fichero de Test

In [158]:
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [159]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

✅ GPU detectada. Trabajando con: "NVIDIA L4"
Memoria VRAM total disponible: 23.66 GB


### 1. Carga del mejor modelo desde drive

In [160]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [161]:
"""test_df = pd.read_csv('/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv')

# Limpiamos el índice extra (si existe) y renombramos la columna
if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)"""

campo_texto = 'text'
test_df[campo_texto] = test_df[campo_texto].str.lower()

In [162]:
model_checkpoint = 'FacebookAI/roberta-base'
model_name = model_checkpoint.split("/")[-1]

#output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Transformers/' + model_name
output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/' + ETIQUETA_OBJETIVO + "/" + resultado
model_output_dir = f"{output_dir}/modelo_{model_name}"

print(f"Cargando tokenizador y mejor modelo desde: {model_output_dir}")
tokenizer_test = AutoTokenizer.from_pretrained(model_output_dir)
model_test = AutoModelForSequenceClassification.from_pretrained(model_output_dir).to(device)
model_test.eval()

Cargando tokenizador y mejor modelo desde: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/MISOGYNY-NON-SEXUAL-VIOLENCE/Roberta_Task3.3_MISOGYNY-NON-SEXUAL-VIOLENCE/modelo_roberta-base


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

### 2. Predicciones sobre el conjunto de test fijo

In [163]:
predicciones = []
y_true = test_df['label'].tolist()
y_pred = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Inferencia en Test ({model_name})"):
    # Usamos la variable campo_texto que definiste antes
    texto = str(row[campo_texto])

    # Tokenización
    inputs = tokenizer_test(texto, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model_test(**inputs)
        # Softmax para obtener la probabilidad de la clase 1 (Misógino)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        prob_misogino = probs[1].item()

    predicciones.append({
        "id_EXIST": row["id_EXIST"],
        "prob_texto": prob_misogino
    })

    # Decisión binaria (Umbral 0.5)
    y_pred.append(1 if prob_misogino > 0.5 else 0)

Inferencia en Test (roberta-base): 100%|██████████| 230/230 [00:02<00:00, 102.45it/s]


### 3. Resultados oficiales y métricas

In [164]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS OFICIALES EN TEST FIJO: {model_name} con el dataset {resultado} ")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS OFICIALES EN TEST FIJO: roberta-base con el dataset Roberta_Task3.3_MISOGYNY-NON-SEXUAL-VIOLENCE 
F1-Score (Macro): 0.4511
Accuracy: 0.8217

Matriz de Confusión:
 [[189   0]
 [ 41   0]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.82      1.00      0.90       189
    Misógino       0.00      0.00      0.00        41

    accuracy                           0.82       230
   macro avg       0.41      0.50      0.45       230
weighted avg       0.68      0.82      0.74       230



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 4. Guardado del CSV para futuro ensemble

In [165]:
ruta_guardado = f"{output_dir}/predicciones_test_{model_name}_{ETIQUETA_OBJETIVO}.csv"
pd.DataFrame(predicciones).to_csv(ruta_guardado, index=False)
print(f"\n✅ Archivo CSV de predicciones guardado en: {ruta_guardado}")


✅ Archivo CSV de predicciones guardado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/MISOGYNY-NON-SEXUAL-VIOLENCE/Roberta_Task3.3_MISOGYNY-NON-SEXUAL-VIOLENCE/predicciones_test_roberta-base_MISOGYNY-NON-SEXUAL-VIOLENCE.csv
